# SenticDiT — Kaggle runner

Reproduces every result in *Revisiting Emotional TTS Fine-Tuning on MELD: What Actually Matters*.
All code lives in the [SenticDiT repo](https://github.com/HellDragger/SenticDiT); this notebook only
fetches it, sets the configuration, and runs the four experiments. Each one runs in its own Python
process, the same as running the original notebooks as separate Kaggle sessions:

| experiment | original notebook | time on 2× T4 |
|---|---|---|
| `v5` | `senticdit_v5.ipynb` — main run (resumed), ablations (skipped), full held-out evaluation | ~2 h |
| `configd` | `senticdit_configd.ipynb` — data quantity at fixed compute + MELD register check | ~4.7 h |
| `addon` | `senticdit_addon.ipynb` — CREMA-D cloned-voice MCD + second-opinion metrics | ~1.6 h |
| `crossclone` | `senticdit_crossclone.ipynb` — 2×2 cross-speaker cloning on CREMA-D | ~35 min |

All four take ~9 h, which fits in one 12 h session. To split them across sessions, switch
experiments off in `RUN`.

**Kaggle setup:** GPU T4 ×2 · Internet **on** · inputs:
`zaber666/meld-dataset`, `aryansharma26/meld-audio`, `aryansharma26/senticdit-r-16` (fine-tuned adapter),
`ejlok1/cremad`, and the earlier `aryansharma26/senticdit` notebook output (main-run checkpoint for `v5`).

## 0 · Fetch the code & install dependencies

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/HellDragger/SenticDiT.git"
REPO_REF = "main"                          # branch or tag to run
REPO_DIR = "/kaggle/working/SenticDiT"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
sys.path.insert(0, REPO_DIR)

from senticdit.setup_env import prepare_environment
prepare_environment()   # pip installs + clones LongCat-AudioDiT (the `audiodit` package)

## 1 · Configuration

`COMMON` applies to every experiment; each experiment's dict overrides it. Any field of
`senticdit/config.py` can be set here (it documents every field). The values below are the ones
the original runs used.

In [ ]:
RUN = {"v5": True, "configd": True, "addon": True, "crossclone": True}

COMMON = dict(
    kaggle_input_dir = "/kaggle/input/datasets/zaber666/meld-dataset/MELD-RAW/MELD.Raw",
    audio_dir        = "/kaggle/input/datasets/aryansharma26/meld-audio/MELD_audio",
    output_dir       = "/kaggle/working/audiodit_fixed",
    seed         = 42,
    eval_seed    = 1234,     # every evaluation block reseeds from this
    cfg_strength = 2.0,      # fixed by fiat; the CFG sweep is reported as a null
    cfm_steps    = 16,
    cremad_dir   = None,     # None = find CREMA-D's AudioWAV folder under /kaggle/input
)

V5 = dict(
    run_tag = "v4_run1",
    # main run: resume the finished 16,860-step checkpoint instead of retraining (~16 h)
    resume_search_dir = "/kaggle/input/notebooks/aryansharma26/senticdit/audiodit_fixed/checkpoints/r16_attn_only",
    skip_ablations              = True,
    best_lora_rank_override     = 16,    # cosmetic when resuming — the checkpoint carries the rank
    best_include_adaln_override = False,
)

CONFIGD = dict(
    run_tag = "v5_configd",
    config_d_max_steps     = 2000,
    config_d_rank          = 16,
    config_d_include_adaln = True,
    configd_resume_root    = None,   # folder with configd_<arm>/step_* to resume an interrupted run
)

ADDON = dict(
    run_tag = "addon",
    adapter_dir = "/kaggle/input/datasets/aryansharma26/senticdit-r-16/r16_attn_adaln/step_016860/lora_adapter",
    cremad_actors           = 24,
    addon_run_part_a        = True,   # CREMA-D cloned-voice MCD
    addon_run_part_b        = True,   # MELD second-opinion metrics
    meld_anchor_per_emotion = 10,
    v5_dir                  = None,   # folder with v5's health_metrics_final.csv for a determinism check
)

CROSSCLONE = dict(
    cremad_actors = 24,
)

## 2 · Run

Each experiment's log streams below and is saved to `<output_dir>/log_<experiment>.txt`; results
land in `output_dir` (`addon/` and `crossclone/` subfolders for those two), plus a
`<experiment>_results.zip` in `/kaggle/working`.

In [ ]:
from senticdit.launcher import run_experiment

for name, overrides in [("v5", V5), ("configd", CONFIGD), ("addon", ADDON), ("crossclone", CROSSCLONE)]:
    if RUN[name]:
        run_experiment(name, {**COMMON, **overrides})